# DINO Linear Probe

Which test errors remain after fitting only the readout?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/timm/vit_small_patch16_224.dino

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "timm==1.0.22"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os
import random
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
# Load the ViT-Small model trained with DINO
model_name = 'vit_small_patch16_224.dino'
model = timm.create_model(model_name, pretrained=True, num_classes=0)  # num_classes=0 → feature extractor
model = model.to(device)
model.eval()

print(f"✅ Model loaded: {model_name}")
print(f"   - Patch size: 16x16")
print(f"   - Input size: 224x224")
print(f"   - Embedding dim: {model.embed_dim}")
print(f"   - Number of heads: {model.blocks[0].attn.num_heads}")
print(f"   - Number of layers: {len(model.blocks)}")

# Check the number of parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"   - Total parameters: {total_params / 1e6:.1f}M")

In [ ]:
# Use the model-specific transform provided by timm
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

print("📐 Transform pipeline:")
print(transform)

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
class DogCatDataset(Dataset):
    """Dog vs Cat image dataset"""
    def __init__(self, img_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.class_names = ['cat', 'dog']
        
        for label_idx, label in enumerate(self.class_names):
            class_dir = os.path.join(img_dir, label)
            if not os.path.exists(class_dir):
                continue
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(class_dir, fname), label_idx))
        
        print(f"   Dataset: {len(self.samples)} images ({self.class_names})")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, path

print("📂 Building dataset:")
dataset = DogCatDataset(IMG_DIR, transform=transform)

In [ ]:
@torch.no_grad()
def extract_features(model, dataset, device, batch_size=16):
    """
    Extract image features with DINO ViT.
    The CLS token output is used as the feature vector.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    all_features = []
    all_labels = []
    all_paths = []
    
    model.eval()
    for images, labels, paths in tqdm(loader, desc='Extracting features'):
        images = images.to(device)
        features = model(images)  # (B, embed_dim) — CLS token output
        
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())
        all_paths.extend(paths)
    
    features = np.concatenate(all_features, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    
    print(f"\n✅ Feature extraction complete!")
    print(f"   - Feature shape: {features.shape}")
    print(f"   - Labels shape: {labels.shape}")
    
    return features, labels, all_paths

features, labels, paths = extract_features(model, dataset, device)

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    features, labels, paths,
    test_size=0.3,
    random_state=SEED,
    stratify=labels
)

print(f"📊 Train/Test Split:")
print(f"   Train: {len(X_train)} samples (Dog: {(y_train==1).sum()}, Cat: {(y_train==0).sum()})")
print(f"   Test:  {len(X_test)} samples (Dog: {(y_test==1).sum()}, Cat: {(y_test==0).sum()})")

## Demo code

Which test errors remain after fitting only the readout?

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
prediction = clf.predict(X_test)
print(accuracy_score(y_test, prediction))
print(confusion_matrix(y_test, prediction))

In [ ]:
# Linear Probing: Logistic Regression (= 1-layer linear classifier)
print("🔧 Training linear probe...\n")

clf = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    C=1.0,          # regularization
    solver='lbfgs'
)
clf.fit(X_train, y_train)

# Evaluate
train_acc = clf.score(X_train, y_train)
test_acc = clf.score(X_test, y_test)
y_pred = clf.predict(X_test)

print(f"{'='*50}")
print(f"  📈 Linear Probing Results")
print(f"{'='*50}")
print(f"  Train Accuracy: {train_acc:.1%}")
print(f"  Test Accuracy:  {test_acc:.1%}")
print(f"{'='*50}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Cat 🐱', 'Dog 🐶']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Cat', 'Dog'])
axes[0].set_yticklabels(['Cat', 'Dog'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20, fontweight='bold',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# --- Accuracy Bar Chart ---
bars = axes[1].bar(['Train', 'Test'], [train_acc, test_acc], 
                    color=['#4CAF50', '#2196F3'], width=0.5, edgecolor='black')
axes[1].set_ylim(0, 1.15)
axes[1].set_title('Linear Probing Accuracy', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Accuracy')
for bar, acc in zip(bars, [train_acc, test_acc]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{acc:.1%}', ha='center', fontsize=16, fontweight='bold')
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions on test samples
n_show = 10
indices = random.sample(range(len(X_test)), min(n_show, len(X_test)))

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('🔍 Test Set Predictions (Linear Probing)', fontsize=16, fontweight='bold')

class_names = ['Cat 🐱', 'Dog 🐶']

for i, idx in enumerate(indices):
    ax = axes[i // 5, i % 5]
    img = Image.open(paths_test[idx]).convert('RGB')
    ax.imshow(img)
    
    true_label = class_names[y_test[idx]]
    pred_label = class_names[y_pred[idx]]
    correct = y_test[idx] == y_pred[idx]
    
    color = 'green' if correct else 'red'
    symbol = '✓' if correct else '✗'
    ax.set_title(f'{symbol} Pred: {pred_label}\nTrue: {true_label}', 
                color=color, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Interpretation

Measured on the fixed Oxford Pets subset, not a general accuracy claim.

Compare the output with the evidence in the input. A generated answer can be wrong.